<h1>Chapter 3 - Loading Data</h1>
<i>Designing loading pipelines for various kinds of file types.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch03_loading_data/loading_data_to_RAG.ipynb)

---

This notebook is for Chapter 3 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisites

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [2]:
!pip install python-docx==1.1.2
!pip install unstructured==0.17.2
!pip install python-magic-bin==0.4.14
!pip install pandas==2.2.3
!pip install PyPDF2==3.0.1
!pip install pillow==11.2.1
!pip install openpyxl==3.1.5
!pip install pdf2image==1.17.0
!pip install pytesseract==0.3.13
!pip install openai==1.82.1
!pip install python-dotenv==1.1.0
!pip install sqlalchemy==2.0.41
!pip install psycopg2-binary==2.9.10
!pip install moviepy==2.2.1
!pip install pdfminer.six==20250506
!pip install pi-heif==0.22.0
!pip install unstructured-inference==1.0.2


ERROR: Could not find a version that satisfies the requirement python-magic-bin==0.4.14 (from versions: none)
ERROR: No matching distribution found for python-magic-bin==0.4.14


### Install Poppler

To use `pdf2image` you need to install `Poppler`.

**Installation instructions:**
- **Google Colab / Ubuntu / Debian:** Run the cell below
- **macOS:** Run `brew install poppler` in your terminal
- **Windows:** Download from [poppler-windows releases](https://github.com/oschwartz10612/poppler-windows/releases/) and add to PATH

In [3]:
import sys

# Check if running in Google Colab or Linux environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !apt-get update -qq
    !apt-get install -y -qq poppler-utils
    print("✓ Poppler installed successfully")
else:
    print("⚠ Running locally. Please install Poppler manually:")
    print("  - macOS: brew install poppler")
    print("  - Ubuntu/Debian: sudo apt-get install poppler-utils")
    print("  - Windows: Download from https://github.com/oschwartz10612/poppler-windows/releases/")

⚠ Running locally. Please install Poppler manually:
  - macOS: brew install poppler
  - Ubuntu/Debian: sudo apt-get install poppler-utils
  - Windows: Download from https://github.com/oschwartz10612/poppler-windows/releases/


### Load sample files

This notebook uses sample Word and PDF files from the `datasets` directory.

When running on Google Colab, the code below will download the datasets from GitHub.

In [4]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("✓ Datasets downloaded to /content/datasets")
else:
    print("⚠ Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [5]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### 1.1 Loading Word Files in Python

Option 1: load word files using the `python_docx` library

In [6]:
import os
import requests
from docx import Document
from io import BytesIO

file_path = "../datasets/word_files/2023_Jan_7_Feature_Engineering_Techniques.docx"
doc = Document(file_path)

text = []
for paragraph in doc.paragraphs:
    text.append(paragraph.text)

full_text = "\n".join(text)


In [7]:
full_text

'\n7 of the Most Used Feature Engineering Techniques\nHands-on Feature Engineering with Scikit-Learn, Tensorflow, Pandas and Scipy\n7 of the most used Feature Engineering Techniques\u200a—\u200aImage by the author\n\nTable of content\nIntroduction\n1. Encoding\n 1.1 Label Encoding using Scikit-learn\n 1.2 One-Hot Encoding using Scikit-learn, Pandas and Tensorflow\n2. Feature Hashing\n 2.1 Feature Hashing using Scikit-learn\n3. Binning / Bucketizing\n 3.1 Bucketizing using Pandas\n 3.2 Bucketizing using Tensorflow\n 3.3 Bucketizing using Scikit-learn\n4. Transformer\n 4.1 Log-Transformer using Numpy\n 4.2 Box-Cox Function using Scipy\n5. Normalize / Standardize\n 5.1 Normalize and Standardize using Scikit-learn\n6. Feature Crossing\n 6.1 Feature Crossing in Polynomial Regression\n 6.2 Feature Crossing and the Kernel-Trick\n7. Principal Component Analysis (PCA)\n 7.1 PCA using Scikit-learn\nSummary\nReferences\n\nIntroduction\nFeature engineering describes the process of formulating rele

Option 2: load word files using the unstructured library

In [8]:
from unstructured.partition.docx import partition_docx
import os
import pandas as pd

# elements = partition_docx(filename=file_path)
elements = partition_docx(filename=file_path)

list_of_elements = []

for element in elements:
    element_dict = {
        "element_id": element.id,
        "file_path": file_path,
        "category": element.category,  # e.g. "Title", "NarrativeText", "ListItem"
        "text": element.text,
        "last_modified": element.metadata.last_modified,
    }

    list_of_elements.append(element_dict)

elements_df = pd.DataFrame(list_of_elements)

In [9]:
elements_df.head()

,element_id,file_path,category,text,last_modified
0,135f726911a68beceb56d92e2b9d10bc,../datasets/word_files/2023_Jan_7_Feature_Engi...,Title,7 of the Most Used Feature Engineering Techniques,2025-12-13T12:10:09
1,f275447183f11b993f2a87d4b428299b,../datasets/word_files/2023_Jan_7_Feature_Engi...,Title,Hands-on Feature Engineering with Scikit-Learn...,2025-12-13T12:10:09
2,9dacd0881e31b366756a6cc20884f661,../datasets/word_files/2023_Jan_7_Feature_Engi...,NarrativeText,7 of the most used Feature Engineering Techniq...,2025-12-13T12:10:09
3,3bec63fc43107e87aae98bbaf5313196,../datasets/word_files/2023_Jan_7_Feature_Engi...,Title,Table of content,2025-12-13T12:10:09
4,b1b29811f875047fef0bab817d6325c5,../datasets/word_files/2023_Jan_7_Feature_Engi...,UncategorizedText,Introduction,2025-12-13T12:10:09


### 1.2 Loading PDF Files

In [10]:
import PyPDF2
import os
import pandas as pd

file_path = "../datasets/pdf_files/2023_Jan_7_Feature_Engineering_Techniques.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    # Initialize an empty string to store the extracted text
    list_of_pages = []
    page_counter = 1

    for page in reader.pages:
        page_dict = {
            "file_name": reader.metadata.get("/Title"),
            "producer": reader.metadata.get("/Producer"),
            "page_number": page_counter,
            "text": page.extract_text(),
            "images": page.images,
        }

        list_of_pages.append(page_dict)

        page_counter += 1

# Convert the list of pages to a pandas DataFrame
pages_df = pd.DataFrame(list_of_pages)

In [11]:
# Display the first few rows of the DataFrame
pages_df.head()

,file_name,producer,page_number,text,images
0,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,1,7\nof\nthe\nMost\nUsed\nFeature\nEngineering\n...,"[File(name=X7.png, data: 2.2 kB)]"
1,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,2,3.2\nBucketizing\nusing\nTensorflow\n3.3\nBuck...,[]
2,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,3,A\nstandard\nMachine\nLearning\npipeline — Ins...,"[File(name=X17.png, data: 692 Byte)]"
3,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,4,"●\nI\nn\nthe\nsupply\nchain\ncontext\n,\nevery...","[File(name=X20.png, data: 2.6 kB)]"
4,2023_Jan_7_Feature_Engineering_Techniques,Skia/PDF m131 Google Docs Renderer,5,Once\nwe\nhave\nenough\ndata\nthat\ndescribes\...,"[File(name=X26.png, data: 1.5 kB)]"


### 1.3 Loading and Handling CSV and Excel Files

In [12]:
###########################################################################################################
# Define the file path to the Word document
###########################################################################################################
# tag::create_additional_table_column[]
import os
import pandas as pd

file_path = "../datasets/csv_files/census-income.xlsx"
df_excel = pd.read_excel(io=file_path)


def create_text_description_of_row(row):
    row["text_description"] = (
        f"""The candidate {row['age']} years old is working in the
            {row['workclass']} sector. The candidate was born in
            {row['native-country']}, is {row['marital-status']}
            and has a {row['relationship']} relationship.
            The candidate has a {row['education']} degree
            and is working as a {row['occupation']}.
            The income of the candidate is {row['income']}."""
    )

    return row


# Apply the function create_text_description_of_row to each row of the data frame
df_extended = df_excel.apply(create_text_description_of_row, axis=1)
# end::create_additional_table_column[]


In [13]:
# Display the first 5 text_description of the dataset
df_extended["text_description"].head()

0    The candidate 39 years old is working in the\n...
1    The candidate 50 years old is working in the\n...
2    The candidate 38 years old is working in the\n...
3    The candidate 53 years old is working in the\n...
4    The candidate 28 years old is working in the\n...
Name: text_description, dtype: object

In [14]:
df_extended["text_description"][0]

'The candidate 39 years old is working in the\n            State-gov sector. The candidate was born in\n            United-States, is Never-married\n            and has a Not-in-family relationship.\n            The candidate has a Bachelors degree\n            and is working as a Adm-clerical.\n            The income of the candidate is <=50K.'

### 1.4 Querying a PostgreSQL Database

This section needs a PostgreSQL database to connect to.

```
CREATE USER rag_user WITH PASSWORD 'raguserpassword123';
GRANT ALL ON ALL TABLES IN SCHEMA public TO rag_user;
```

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# PostgreSQL connection parameters
username = "rag_user"
password = "raguserpassword123"
host = "localhost"
port = "5432"
database = "postgres"

connection_string = (
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)
engine = create_engine(connection_string)

with engine.connect() as connection:
    query = """SELECT * FROM categories ORDER BY category_id ASC """
    result = pd.read_sql(query, connection)
    print(result)

NameError: name 'username' is not defined

### 1.5 Loading Audio Files by Using Speech-to-Text Models

In [16]:
import os
from openai import OpenAI

audio_file_path = "../datasets/audio_files/harvard.wav"

# initialize the OpenAI client with your API key
client = OpenAI()

with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1", file=audio_file
    )

transcription

Transcription(text='The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.', logprobs=None, usage={'type': 'duration', 'seconds': 19})

### 1.6 Extracting Text from Images and PDFs Using OCR

In [17]:
import os
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

image = Image.open(
    fp="../datasets/images/example_finance_reporting_slide.png"
)

text = pytesseract.image_to_string(image)

TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

In [18]:
import os
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

file_path = (
    "../datasets/images/"
    "2023_Jan_7_Feature_Engineering_Techniques.pdf"
)

images = convert_from_path(pdf_path=file_path)

text = []
for i, image in enumerate(images):
    page_text = pytesseract.image_to_string(image)
    text.append(page_text)

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

### 1.7 Extracting Text from Images using Multimodal Models

In [40]:
import os
import base64
from openai import OpenAI

png_file_path = "../datasets/images/example_finance_reporting_slide.png"

# initialize the OpenAI client
client = OpenAI()

with open(png_file_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "Extract the text from the image attached. Make sure to only "
        "extract only the text. If there is no text in the image, "
        "please return with the sentence 'No text found in the image."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # define the model to use
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=500,
    )

    content = response.choices[0].message.content
    print(content)

Sales volume growth driven by EMEA 18/19

Business Development

- Sales volume growth driven by EMEA with sales activity up to 2.3x YoY
- Sales activity in 20 countries during Q2 19
  - US (28%), Spain (20%), India (15%) and Norway (11%) are the main contributors to the Q2 19 sales volume


### 1.8 Generating Text Summaries for Images Using Multimodal Models

In [41]:
import os
import base64
from openai import OpenAI

image_path = "../datasets/images/vietnam.png"

# initialize the OpenAI client
client = OpenAI()

with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "You are an assistant for visually impaired users. "
        "Describe the image in detail."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=150,
    )

    content = response.choices[0].message.content
    print(content)

The image depicts a modern urban skyline at dusk, with a serene body of water in the foreground reflecting the buildings and the sky. 

On the left side, there is a dock with a white and yellow boat, suggesting leisure activities or transportation. The water appears calm, mirroring the skyline above, creating a peaceful atmosphere. 

The skyline is dominated by several high-rise buildings, showcasing a variety of architectural styles. The tallest building, which has a sharp peak and sleek glass exterior, stands prominently in the center. Other notable buildings have unique shapes, including one with a curved top and another with a distinctive curved cutout. 

As the sun sets, the sky transitions from blue to a lighter purplish hue, with soft clouds scattered across.


### 1.9 Generating Text Summaries for Embedded Tables Using Multimodal Models

The following code will use OCR and by default will use `unstructured_pytesseract`. Run the following to install `tessearct-ocr`.

In [35]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils

!pip install -q "unstructured[pdf]" unstructured_pytesseract


E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [36]:
import os
from unstructured.partition.pdf import partition_pdf

pdf_file_path = "../datasets/pdf_files/adult_data_article.pdf"

tables = []
texts = []

# partition the PDF file into its elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
)

for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        tables.append(str(element))

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [ ]:
from openai import OpenAI
import pandas as pd


def summarize_tables(row):
    summary_prompt = (
        f"You are an assistant tasked with summarizing tables. "
        f"Give a concise summary of the table. "
        f"Table chunk: {row.table}"
    )

    # Initialize the OpenAI API client and generate table summary
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": summary_prompt}],
        temperature=0.7,
        max_tokens=150,
    )

    row["table_summary"] = response.choices[0].message.content

    return row


# create a pandas dataframe from the tables
tables_df = pd.DataFrame(tables, columns=["table"])

# add a column to the dataframe to store the summaries
tables_df = tables_df.apply(summarize_tables, axis=1)

NameError: name 'tables' is not defined

In [38]:

# tag::test_ask_a_question[]
# define a random question to the embedded table
user_question = "What are the education levels of the people working in Sales?"


def build_prompt_and_generate_answer(user_question, found_table):
    """
    This function builds a prompt using the user's question and the context of the table
    and generates an answer using the OpenAI API

    Parameters:
        user_question: the question asked by the user
        found_table: the table context to generate the answer from

    Returns:
        answered_question: the answer to the user's question
    """

    question_prompt = f"""You are an assistant using the content from PDFs \
                        to answer questions. Below you can find the \
                        user's question and relevant context. Please use the \
                        context to generate an answer to the user's question.

                        # User question: {user_question}

                        # Context:

                        ## Table summary:
                        {found_table.table_summary}

                        ## Table content:
                        {found_table.table}""".stripe()

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    answered_question = (
        client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": question_prompt}],
            temperature=0.7,
            max_tokens=150,
        )
        .choices[0]
        .message.content
    )

    return answered_question


# generate the answer to the user's question
# as context we using the first entry in the tables_df
answered_question = build_prompt_and_generate_answer(
    user_question=user_question, found_table=tables_df.iloc[0]
)

print(answered_question)
# end::test_ask_a_question[]

NameError: name 'tables_df' is not defined

### 1.10 Parsing PDFs with Multiple Media Content Using Unstructured and Multimodal Models

In [39]:
from unstructured.partition.pdf import partition_pdf
import os

# set the OCR agent to tesseract
os.environ["OCR_AGENT"] = "tesseract"

pdf_file_path = "../datasets/pdf_files/adult_data_article.pdf"
image_output_dir = "../datasets/extracted_content_from_pdfs/images"

# create output directory if it doesn't exist
os.makedirs(image_output_dir, exist_ok=True)

# get elements using the function extract_pdf_elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=image_output_dir,
)

# categorize elements by type
tables = []
texts = []
titles = []

# fill the just created lists with the elements
for element in raw_pdf_elements:
    element_type = str(type(element))
    if "unstructured.documents.elements.Table" in element_type:
        tables.append(str(element))
    elif "unstructured.documents.elements.NarrativeText" in element_type:
        texts.append(str(element))
    elif "unstructured.documents.elements.Title" in element_type:
        titles.append(str(element))

ImportError: libGL.so.1: cannot open shared object file: No such file or directory

### 1.11 Loading Videos Using Speech-to-Text and Multimodal Models

You can find the test video I used on YouTube: [Learn Data Science Tutorial - Full Course for Beginners](https://www.youtube.com/watch?v=ua-CiDNNj30)

In [ ]:
import os
import pandas as pd

from moviepy import VideoFileClip, TextClip, CompositeVideoClip

video_file_path = "../datasets/videos/learn-data-science-tutorial.mp4"
image_output_folder = "../datasets/videos/video_extracted_images"

# create output folder if it doesn't exist
os.makedirs(image_output_folder, exist_ok=True)

clip = VideoFileClip(video_file_path)

# create a list of timestamps from which to extract a frame
time_step = 10  # time in seconds
timestamps = list(range(0, int(clip.duration) - time_step, time_step))

# for each timestamp extract a frame
for timestamp in timestamps:
    frame_image_path = os.path.join(
        image_output_folder, f"frame_{timestamp}.png"
    )
    clip.save_frame(frame_image_path, t=timestamp)

In [ ]:
# for each timestamp extract the audio sequence and save it to a .mp3 file
audio_output_folder = "../datasets/videos/video_extracted_audio"

# create output folder if it doesn't exist
os.makedirs(audio_output_folder, exist_ok=True)

for timestamp in timestamps:
    audio_clip = clip.subclip(timestamp, timestamp + time_step).audio
    output_audio_path = os.path.join(
        audio_output_folder, f"audio_{timestamp}.mp3"
    )
    audio_clip.write_audiofile(output_audio_path)

In [ ]:

# tag::audio_to_text[]
from openai import OpenAI


def audio_to_text(audio_path):
    """
    Convert audio to text using OpenAI's Whisper model.

    Parameters:
    audio_path (str): The path to the audio file.

    Returns:
    str: The text recognized from the audio.

    """
    # Initialize the OpenAI client with your API key

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    # Open and read the audio file
    with open(audio_path, "rb") as audio_file:
        # Transcribe
        transcription = client.audio.transcriptions.create(
            model="whisper-1", file=audio_file
        )

    # save the transcription to a text file
    text_file_path = audio_path.replace(".mp3", ".txt")
    with open(text_file_path, "w") as text_file:
        text_file.write(transcription.text)

    return


# List all files in folder audio_output_folder
audio_files = os.listdir(audio_output_folder)

for audio_file in audio_files:
    absolut_path_audio_file = os.path.join(audio_output_folder, audio_file)
    # Use the function audio_to_text to convert the audio to text
    audio_to_text(audio_path=absolut_path_audio_file)
# end::audio_to_text[]